In [10]:
import numpy as np
from scipy.stats import multivariate_t
from limvam.directlingam_extension import find_parent_variable

# generate data

In [11]:
# parameters
m = 8
p = 6
n = 1000
random_state = 20
rng = np.random.RandomState(random_state)
disturbances = "student_t"
n_runs = 100

In [12]:
def generate_data(
    m,
    p,
    n,
    rng=None,
    disturbances="gaussian",
):
    """
    Generate data according to the model xi = Bi xi + ei, where the Bi are DAG matrices 
    and the disturbances ei are correlated across views.
    """
    # variance of the disturbances
    M = rng.randn(p, m, m)
    Sigmas = np.zeros((p, m, m))
    for j in range(p):
        Sigmas[j] = M[j] @ M[j].T + m * np.eye(m)
    
    # disturbances
    E = np.zeros((m, p, n))
    for j in range(p):
        if disturbances == "gaussian":
            E[:, j] = rng.multivariate_normal(mean=np.zeros(m), cov=Sigmas[j], size=(n,)).T
        elif disturbances == "student_t":
            E[:, j] = multivariate_t.rvs(loc=np.zeros(m), shape=Sigmas[j], df=1, size=n, random_state=rng).T
        else:
            raise ValueError("The parameter 'disturbances' should be 'gaussian' or 'student_t'.")

    # strictly lower triangular matrices T
    T = rng.normal(size=(m, p, p))
    for i in range(m):
        T[i][np.triu_indices(p, k=0)] = 0
    
    # causal order P
    order = rng.permutation(p)
    P = np.eye(p)[order]

    # causal effect matrices B
    B = P.T @ T @ P
    
    # mixing matrices
    A = np.linalg.inv(np.eye(p) - B)
    
    # observations
    X = np.array([Ai @ Ei for Ai, Ei in zip(A, E)])
    return X, B, T, P, order

# DirectLiNGAM extension

In [13]:
# percentage of runs where the parent variable is found
success = 0

for _ in range(n_runs):
    X, _, _, _, order = generate_data(m, p, n, rng, disturbances)
    estimated_first_vb = find_parent_variable(X)
    success += (order[0] == estimated_first_vb)
success /= n_runs

In [16]:
print(f"Percentage of runs where the parent variable is found : {success*100:.2f} %")
print(f"Chance level : {100/p:.2f} %")

Percentage of runs where the parent variable is found : 22.90 %
Chance level : 16.67 %
